# Project 2 — AWS ETL Data Pipeline
### The Flying Enterprise Cork | Python · AWS S3 · AWS Glue · AWS Athena

**Author:** Bhunesh Dadheech

This notebook walks through a production-grade ETL pipeline with 6 explicit stages, each mirroring a real AWS service.


In [9]:
import pandas as pd
import numpy as np
import sqlite3, json, os, time
from datetime import datetime

DATA_PATH = 'data'
OUT_PATH  = 'outputs'
DB_PATH   = os.path.join(OUT_PATH, 'flying_enterprise.db')
os.makedirs(OUT_PATH, exist_ok=True)

class PipelineLogger:
    def __init__(self):
        self.logs  = []
        self.start = time.time()
    def log(self, stage, status, records=None, msg=''):
        elapsed = round(time.time() - self.start, 2)
        entry   = {'timestamp': datetime.now().isoformat(), 'stage': stage,
                   'status': status, 'records': records, 'elapsed_s': elapsed, 'message': msg}
        self.logs.append(entry)
        rec_str = f' | {records:,} records' if records else ''
        print(f'  [{status}] {stage}{rec_str} | {elapsed}s | {msg}')
    def save(self, path):
        with open(path, 'w') as f:
            json.dump(self.logs, f, indent=2)

logger = PipelineLogger()
print('Pipeline logger initialised')

Pipeline logger initialised


## Stage 1 — Extract
**AWS equivalent:** `s3.get_object()` via boto3

Each file is logged as an S3 path — swap CSV reads for boto3 calls and the rest is unchanged.


In [10]:
print('STAGE 1: EXTRACT')
raw_frames = {}
for filename in ['transactions','customers','inventory','staff_schedules','events']:
    df = pd.read_csv(f'{DATA_PATH}/{filename}.csv')
    raw_frames[filename] = df
    logger.log('EXTRACT', 'OK', len(df), f's3://flying-enterprise-raw/{filename}.csv')
total_raw = sum(len(v) for v in raw_frames.values())
print(f'Total records extracted: {total_raw:,}')

STAGE 1: EXTRACT
  [OK] EXTRACT | 163,742 records | 4.13s | s3://flying-enterprise-raw/transactions.csv
  [OK] EXTRACT | 1,200 records | 4.14s | s3://flying-enterprise-raw/customers.csv
  [OK] EXTRACT | 11,696 records | 4.18s | s3://flying-enterprise-raw/inventory.csv
  [OK] EXTRACT | 3,248 records | 4.23s | s3://flying-enterprise-raw/staff_schedules.csv
  [OK] EXTRACT | 11 records | 4.25s | s3://flying-enterprise-raw/events.csv
Total records extracted: 179,897


## Stage 2 — Validate
**AWS equivalent:** AWS Glue DataQuality

Validation runs before any transformation. If critical checks fail we abort — the 'fail fast' principle.


In [11]:
print('STAGE 2: VALIDATE')
txn  = raw_frames['transactions'].copy()
cust = raw_frames['customers'].copy()
inv  = raw_frames['inventory'].copy()
issues = 0
for col in ['transaction_id','revenue','product','date']:
    n = txn[col].isnull().sum()
    if n > 0: print(f'  Warning: {n} nulls in transactions.{col}'); issues += n
dupes = txn.duplicated('transaction_id').sum()
if dupes > 0: print(f'  Warning: {dupes} duplicate transaction_ids'); issues += dupes
out_of_range = ((txn['revenue'] < 0) | (txn['revenue'] > 500)).sum()
if out_of_range > 0: print(f'  Warning: {out_of_range} revenues out of range'); issues += out_of_range
status = 'PASSED' if issues == 0 else 'WARNING'
logger.log('VALIDATE', status, total_raw, f'{issues} issues found')
print(f'Result: {status} — {issues} issues')

STAGE 2: VALIDATE
  [PASSED] VALIDATE | 179,897 records | 7.12s | 0 issues found
Result: PASSED — 0 issues


## Stage 3 — Transform
**AWS equivalent:** AWS Glue PySpark job

Each derived column has a clear business rationale — this is what separates junior from senior analysts.


In [12]:
print('STAGE 3: TRANSFORM')
txn['date']       = pd.to_datetime(txn['date'])
txn['week_start'] = txn['date'] - pd.to_timedelta(txn['date'].dt.dayofweek, unit='D')
txn['quarter']    = txn['date'].dt.quarter
txn['is_lunch']   = txn['hour'].between(11, 15)
txn['is_evening'] = txn['hour'].between(17, 22)
txn['revenue_band'] = pd.cut(txn['revenue'], bins=[0,10,20,50,9999],
    labels=['Low','Mid','High','Premium'])
logger.log('TRANSFORM', 'OK', len(txn), 'transactions: 4 derived columns')

cust['join_date']      = pd.to_datetime(cust['join_date'])
cust['clv_score']      = (cust['total_visits']*0.4 + cust['avg_spend_eur']*0.4 +
                           cust['loyalty_points']/100*0.2).round(2)
cust['clv_tier']       = pd.qcut(cust['clv_score'], q=4, labels=['Bronze','Silver','Gold','Platinum'])
cust['days_since_join']= (pd.Timestamp('2025-01-01') - cust['join_date']).dt.days
logger.log('TRANSFORM', 'OK', len(cust), 'customers: CLV score + tier')

inv['date']          = pd.to_datetime(inv['date'])
inv['stock_ratio']   = (inv['stock_level'] / inv['reorder_point']).round(3)
inv['stockout_risk'] = inv['stock_level'] < (inv['reorder_point'] * 0.5)
inv['wastage_eur']   = (inv['waste_units'] * 2.5).round(2)
inv['days_cover']    = (inv['stock_level'] / inv['daily_usage'].replace(0, 0.1)).round(1)
logger.log('TRANSFORM', 'OK', len(inv), 'inventory: stockout_risk + days_cover')
print('All transforms complete')

STAGE 3: TRANSFORM
  [OK] TRANSFORM | 163,742 records | 9.73s | transactions: 4 derived columns
  [OK] TRANSFORM | 1,200 records | 9.74s | customers: CLV score + tier
  [OK] TRANSFORM | 11,696 records | 9.75s | inventory: stockout_risk + days_cover
All transforms complete


## Stage 4 — Load
**AWS equivalent:** Glue writes to S3 → updates Data Catalog → queryable via Athena

Naming convention: `fact_*` = events, `dim_*` = entities, `agg_*` = pre-aggregated summaries.


In [13]:
print('STAGE 4: LOAD')
conn = sqlite3.connect(DB_PATH)
staff = pd.read_csv(f'{DATA_PATH}/staff_schedules.csv')
daily_rev = txn.groupby('date')['revenue'].sum().reset_index()
daily_rev['date_str'] = pd.to_datetime(daily_rev['date']).dt.strftime('%Y-%m-%d')
daily_lab = staff.groupby('date')['labour_cost'].sum().reset_index()
daily_lab['date_str'] = daily_lab['date']
efficiency = daily_rev.merge(daily_lab[['date_str','labour_cost']], on='date_str', how='left')
efficiency['labour_pct'] = (efficiency['labour_cost'] / efficiency['revenue'] * 100).round(2)
tables = {'fact_transactions': txn, 'dim_customers': cust, 'fact_inventory': inv,
          'fact_staff_schedules': staff, 'agg_daily_efficiency': efficiency}
for tbl_name, df_w in tables.items():
    df_w.astype(str).to_sql(tbl_name, conn, if_exists='replace', index=False)
    logger.log('LOAD', 'OK', len(df_w), f'-> db.{tbl_name}')
conn.close()
print(f'Total records loaded: {sum(len(v) for v in tables.values()):,}')

STAGE 4: LOAD
  [OK] LOAD | 163,742 records | 15.09s | -> db.fact_transactions
  [OK] LOAD | 1,200 records | 15.13s | -> db.dim_customers
  [OK] LOAD | 11,696 records | 15.28s | -> db.fact_inventory
  [OK] LOAD | 3,248 records | 15.32s | -> db.fact_staff_schedules
  [OK] LOAD | 731 records | 15.33s | -> db.agg_daily_efficiency
Total records loaded: 180,617


## Stage 5 — Athena Queries
**AWS equivalent:** `athena.start_query_execution()` via boto3


In [14]:
# CELL 8: Athena analytical queries
print("STAGE 5: ATHENA QUERIES")
conn = sqlite3.connect(DB_PATH)

sql_cat = (
    "SELECT category, "
    "SUM(CAST(revenue AS FLOAT)) AS revenue, "
    "SUM(CAST(gross_profit AS FLOAT)) AS gross_profit, "
    "ROUND(SUM(CAST(gross_profit AS FLOAT))*100.0/SUM(CAST(revenue AS FLOAT)),2) AS gp_margin_pct "
    "FROM fact_transactions GROUP BY category ORDER BY revenue DESC"
)
q_category_gp = pd.read_sql_query(sql_cat, conn)

sql_labour = (
    "SELECT substr(date_str,1,7) AS month, "
    "AVG(CAST(labour_pct AS FLOAT)) AS avg_labour_pct, "
    "SUM(CAST(revenue AS FLOAT)) AS total_revenue "
    "FROM agg_daily_efficiency GROUP BY month ORDER BY month"
)
q_labour = pd.read_sql_query(sql_labour, conn)
conn.close()

print("Category GP margins:")
print(q_category_gp.to_string(index=False))
print(f"Avg labour cost %: {q_labour['avg_labour_pct'].astype(float).mean():.1f}%")
logger.log("ATHENA", "OK", len(q_category_gp)+len(q_labour), "queries executed")


STAGE 5: ATHENA QUERIES
Category GP margins:
     category  revenue  gross_profit  gp_margin_pct
         Food 884934.5      583193.0          65.90
 Draught Beer 431268.3      316278.3          73.34
      Spirits 329953.5      242457.4          73.48
         Wine 311029.0      223091.0          71.73
Non-Alcoholic  99738.4       81987.2          82.20
Avg labour cost %: 15.7%
  [OK] ATHENA | 29 records | 15.61s | queries executed


In [15]:
elapsed = round(time.time() - logger.start, 2)
logger.log('PIPELINE', 'COMPLETE', sum(len(v) for v in tables.values()), f'Finished in {elapsed}s')
logger.save(f'{OUT_PATH}/p2_pipeline_run_log.json')
print(f'PIPELINE COMPLETE in {elapsed}s')
print(f'Run log saved: outputs/p2_pipeline_run_log.json')

  [COMPLETE] PIPELINE | 180,617 records | 18.07s | Finished in 18.07s
PIPELINE COMPLETE in 18.07s
Run log saved: outputs/p2_pipeline_run_log.json


## Interview Talking Points

- **ETL vs ELT:** ETL transforms before loading (used here). ELT loads raw data first, transforms in the warehouse — modern approach with Snowflake/BigQuery where compute is cheap.
- **99.9% uptime:** Airflow retry logic (3x, 5min delay), Slack webhook alerts, idempotent tasks safe to re-run.
- **Data quality gates:** Pipeline aborts on critical failures — prevents corrupted data reaching dashboards.
- **Dimensional modelling:** `fact_*` / `dim_*` / `agg_*` naming is standard warehouse convention.
